<a href="https://colab.research.google.com/github/muqadas007-jerry/Loan-default-prediction/blob/main/2ns_task_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from collections import defaultdict

In [4]:
# 1. Load Dataset
df = pd.read_csv("/content/hmeq.csv")  # Replace 'latin-1' with the correct encoding if it's different.

In [6]:
X = df.drop("BAD", axis=1)
y = df["BAD"]

In [7]:
# Handle missing values
numeric_cols = X.select_dtypes(include=["float64", "int64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

In [8]:
# Imputation
X[numeric_cols] = SimpleImputer(strategy="median").fit_transform(X[numeric_cols])
X[categorical_cols] = SimpleImputer(strategy="most_frequent").fit_transform(X[categorical_cols])


In [9]:

# Label encoding for categorical
label_encoders = defaultdict(LabelEncoder)
for col in categorical_cols:
    X[col] = label_encoders[col].fit_transform(X[col])

In [10]:

# Standardize numeric features
X[numeric_cols] = StandardScaler().fit_transform(X[numeric_cols])

In [11]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)


In [12]:
# SMOTE for imbalance
X_train_res, y_train_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

In [13]:
# Train models
lgbm = LGBMClassifier(random_state=42)
svm = SVC(probability=True, random_state=42)

lgbm.fit(X_train_res, y_train_res)
svm.fit(X_train_res, y_train_res)

[LightGBM] [Info] Number of positive: 3817, number of negative: 3817
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002482 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2553
[LightGBM] [Info] Number of data points in the train set: 7634, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


SVC(probability=True, random_state=42)

In [14]:
# Predictions
y_pred_lgbm = lgbm.predict(X_test)
y_pred_svm = svm.predict(X_test)

In [15]:
# Reports
print("LightGBM Report:\n", classification_report(y_test, y_pred_lgbm))
print("SVM Report:\n", classification_report(y_test, y_pred_svm))

LightGBM Report:
               precision    recall  f1-score   support

           0       0.94      0.95      0.95       954
           1       0.80      0.75      0.77       238

    accuracy                           0.91      1192
   macro avg       0.87      0.85      0.86      1192
weighted avg       0.91      0.91      0.91      1192

SVM Report:
               precision    recall  f1-score   support

           0       0.93      0.84      0.88       954
           1       0.54      0.75      0.63       238

    accuracy                           0.82      1192
   macro avg       0.74      0.80      0.76      1192
weighted avg       0.85      0.82      0.83      1192



In [16]:
pip install lightgbm